<a href="https://colab.research.google.com/github/luqthewolf-lgtm/Guia-Defijitivo/blob/main/parte4_definitivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guia Definitivo -- Parte 4 de 5
## Vaga Especifica: Recomendacao + Experimentacao + AWS/SageMaker + Prova 2026

**Pre-requisito:** Partes 1, 2 e 3 concluidas

**Modulos:**
- Modulo A: Sistemas de Recomendacao -- **MUITO ALTA** (vaga menciona diretamente)
- Modulo B: Experimentacao e Testes A/B -- **MUITO ALTA** (vaga menciona diretamente)
- Modulo C: AWS e SageMaker -- **ALTA** (entrevista e trabalho real)
- Modulo D: Prova Estimada 2026 -- **MAXIMA** (simulado completo)

---

## Por que esta parte existe?

A descricao da vaga diz explicitamente:
- 'problemas de **personalizacao, recomendacoes** e propensao'
- 'apoiar iniciativas de **experimentacao** e **medicao de impacto**'
- 'jornadas digitais **hiperpersonalizadas**'

Candidatos que so estudam o ISLP classico erram as questoes especificas da vaga.

---

## Glossario

| Termo | Definicao |
|-------|-----------|
| Filtragem Colaborativa | Recomendacao baseada em usuarios similares |
| Filtragem por Conteudo | Recomendacao baseada nas caracteristicas dos itens |
| SVD | Singular Value Decomposition: matrix factorization |
| Cold Start | Problema de recomendar para usuario sem historico |
| Precision@K | Dos K recomendados, quantos sao relevantes? |
| NDCG@K | Normalized DCG: considera a posicao na lista |
| MRR | Mean Reciprocal Rank: posicao do 1o item relevante |
| Propensao | P(cliente aceita oferta) para cada produto |
| Lift | Quanto melhor que aleatorio o modelo e |
| H0 | Hipotese nula: sem efeito/diferenca |
| MDE | Minimo Efeito Detectavel |
| Peeking | Checar resultado antes do tamanho planejado |
| Guardrail | Metrica que nao pode piorar no experimento |
| DiD | Diferencas em Diferencas: metodo causal |
| PSI | Population Stability Index: mede drift |
| S3 | Amazon Simple Storage Service |
| Athena | Query SQL em dados no S3 |
| SageMaker | Plataforma AWS completa para ML |
| Endpoint | Servico para servir previsoes em tempo real |
| Batch Transform | Scoring de grandes volumes offline |


---
# Setup

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn scipy xgboost mlflow --quiet
print('OK!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings, math, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LogisticRegression, ElasticNet, Ridge
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, mean_squared_error, log_loss
from sklearn.datasets import make_classification
from xgboost import XGBClassifier
import mlflow, mlflow.sklearn

np.random.seed(42)
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.3})
print('Imports OK!')

---
# MODULO A -- Sistemas de Recomendacao
## Prioridade: MUITO ALTA | Especifico da vaga

---

## A dor que os sistemas de recomendacao resolvem

Sem recomendacao: o Itau manda o mesmo email para 3 milhoes de clientes PJ.
Taxa de aceite: 2.3%. 96.7% das ofertas sao irrelevantes.

Com recomendacao: o modelo calcula para cada cliente qual produto ele
tem maior probabilidade de aceitar AGORA.
Taxa de aceite: 6.1%. Lift de 2.65x sobre o baseline.

Isso e o que a vaga pede: construir esses modelos.

---

## A.1 Os tres tipos de filtragem

**Colaborativa:** usuarios similares gostam de itens similares.
Usa comportamento coletivo -- sem precisar conhecer os itens.
- User-based: "clientes como voce contrataram X"
- Item-based: "quem contratou A tambem contratou B"
- Matrix Factorization (SVD): fatores latentes

**Por conteudo:** recomenda itens similares ao que o usuario ja usou,
baseado nas caracteristicas dos itens. Nao precisa de outros usuarios.
Limitacao: nao descobre novos tipos.

**Hibrido:** combina os dois. Padrao em producao.

---

## A.2 Cold Start -- o problema critico

Novo cliente abre conta hoje. Sem historico. Sistema colaborativo falha.

Estrategias:
1. Modelo de propensao com features de cadastro (CNAE, porte, regiao)
2. Produtos mais populares do segmento
3. Regras de negocio por CNAE
4. Onboarding ativo: perguntar o que o cliente precisa

---

## A.3 Metricas

| Metrica | O que mede | Quando usar |
|---------|------------|-------------|
| Precision@K | Qualidade dos K recomendados | Cada recomendacao tem custo |
| Recall@K | Cobertura dos relevantes | Perder item relevante e grave |
| **NDCG@K** | Considera a posicao | Lista ordenada no app -- PRINCIPAL |
| MRR | Posicao do 1o acerto | So o primeiro item importa |

NDCG: item relevante na posicao 1 vale mais que na posicao 5.
Padrao em competicoes de recomendacao.

---

## A.4 Next Best Product (NBP)

Para cada cliente: calcular P(aceita produto X) para todos os X.
Ordenar por probabilidade. Recomendar top-K.

Isso e um modelo de propensao multi-produto.
E XGBoost/Logistica com target binario treinado por produto.


In [ ]:
# A.1 -- MATRIZ USUARIO-ITEM E FILTRAGEM COLABORATIVA
np.random.seed(42)

n_clientes = 200
n_produtos  = 15
produtos = [
    'Conta_PJ', 'Credito_Giro', 'Seguro_Empresa', 'CDB',
    'Cartao_PJ', 'Antecipacao', 'Seguro_Frota', 'FGI',
    'Pix_Cobr', 'Maquininha', 'Folha_Pgto', 'Previdencia',
    'Cambio', 'Trade_Finance', 'Garantia'
]

# Criando matriz espars: NaN = cliente nao usou o produto
matriz_raw = np.full((n_clientes, n_produtos), np.nan)

for i in range(n_clientes):
    n_usados   = np.random.randint(2, 7)                    # 2-6 produtos por cliente
    idx_usados = np.random.choice(n_produtos, n_usados, replace=False)
    for j in idx_usados:
        matriz_raw[i, j] = np.random.choice([3,4,5], p=[0.2,0.3,0.5])  # rating

total     = n_clientes * n_produtos
preench   = (~np.isnan(matriz_raw)).sum()
espars    = 1 - preench / total

print('=== MATRIZ USUARIO-ITEM ===')
print(f'  Dimensoes: {n_clientes} clientes x {n_produtos} produtos')
print(f'  Interacoes: {preench}')
print(f'  Esparsidade: {espars*100:.1f}%  (tipico: 90-99%)')
print()

# Visualizando
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(
    pd.DataFrame(matriz_raw[:40], columns=produtos),
    ax=axes[0], cmap='YlOrRd', linewidths=0.05, linecolor='gray'
)
axes[0].set_title('Matriz Usuario-Item (40 primeiros)\nBranco = nao usou')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=7)

uso = (~np.isnan(matriz_raw)).sum(axis=0)
pd.Series(uso, index=produtos).sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue'
)
axes[1].set_title('Popularidade por produto')
plt.tight_layout(); plt.show()

# Filtragem Colaborativa -- similaridade de cosseno
matriz_bin = np.nan_to_num(matriz_raw, nan=0.0)          # NaN -> 0
norms      = np.linalg.norm(matriz_bin, axis=1, keepdims=True)
norms[norms == 0] = 1
M_norm     = matriz_bin / norms
sim_usr    = M_norm @ M_norm.T                           # sim entre usuarios
sim_itm    = (matriz_bin.T / np.linalg.norm(matriz_bin.T, axis=1, keepdims=True)) @ \
             (matriz_bin.T / np.linalg.norm(matriz_bin.T, axis=1, keepdims=True)).T

def recomendar(uid, n_rec=3):
    sims     = sim_usr[uid].copy()
    sims[uid] = -1                                       # excluir si mesmo
    top_viz  = np.argsort(sims)[::-1][:10]              # 10 mais similares
    ja_usou  = set(np.where(matriz_bin[uid] > 0)[0])
    scores   = np.zeros(n_produtos)
    for v in top_viz:
        scores += sims[v] * (matriz_bin[v] > 0).astype(float)
    for j in ja_usou: scores[j] = -1
    top = np.argsort(scores)[::-1][:n_rec]
    return top, scores[top]

uid = 0
ja  = np.where(matriz_bin[uid] > 0)[0]
rec, sc = recomendar(uid)
print(f'Cliente {uid}:')
print(f'  Ja usou: {[produtos[i] for i in ja]}')
print(f'  Recomendacoes user-based:')
for idx, s in zip(rec, sc):
    print(f'    -> {produtos[idx]:<20} (score={s:.3f})')

print()
print('[!] COLD START: novo cliente sem historico -> filtragem colaborativa falha')
print('    Solucao: modelo de propensao com features de cadastro (CNAE, porte, regiao)')

In [ ]:
# A.2 -- MATRIX FACTORIZATION COM SVD
from scipy.linalg import svd as scipy_svd

# Preencher NaN com media do cliente (necessario para SVD)
M_fill = matriz_raw.copy()
for i in range(n_clientes):
    linha = M_fill[i]
    media = np.nanmean(linha) if not np.all(np.isnan(linha)) else 3.0
    linha[np.isnan(linha)] = media
    M_fill[i] = linha

k_fat = 5    # numero de fatores latentes
U, S, Vt = scipy_svd(M_fill, full_matrices=False)

# Reconstrucao com apenas k fatores
M_prev = U[:, :k_fat] @ np.diag(S[:k_fat]) @ Vt[:k_fat, :]
M_prev = np.clip(M_prev, 1, 5)                  # ratings entre 1 e 5

var_exp = (S[:k_fat]**2) / (S**2).sum() * 100   # variancia por fator

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, k_fat+1), var_exp, color='steelblue')
axes[0].set_title(f'SVD: {k_fat} fatores latentes\nCada fator = um tipo de cliente')
axes[0].set_xlabel('Fator'); axes[0].set_ylabel('Variancia explicada (%)')

# Rating previsto vs real
reais, prev = [], []
for i in range(n_clientes):
    for j in range(n_produtos):
        if not np.isnan(matriz_raw[i,j]):
            reais.append(matriz_raw[i,j])
            prev.append(M_prev[i,j])
axes[1].scatter(reais, prev, alpha=0.3, s=15, color='coral')
axes[1].plot([1,5],[1,5],'k--',lw=2, label='Perfeito')
axes[1].set_xlabel('Rating real'); axes[1].set_ylabel('Rating previsto')
axes[1].set_title('SVD: previsao vs real'); axes[1].legend()
plt.tight_layout(); plt.show()

def rec_svd(uid, n_rec=3):
    ja = set(np.where(~np.isnan(matriz_raw[uid]))[0])
    sc = M_prev[uid].copy()
    for j in ja: sc[j] = -1
    top = np.argsort(sc)[::-1][:n_rec]
    return top, sc[top]

rec_s, sc_s = rec_svd(0)
print('Recomendacoes SVD para cliente 0:')
for i, s in zip(rec_s, sc_s):
    print(f'  -> {produtos[i]:<20} (rating previsto: {s:.2f})')
print()
print(f'Variancia explicada pelos {k_fat} fatores: {var_exp.sum():.1f}%')
print()
print('POR QUE SVD FUNCIONA:')
print('  Fator latente 1 pode capturar: PME digital (alta propensao a Maquininha)')
print('  Fator latente 2 pode capturar: exportador (alta propensao a Cambio)')
print('  Rating previsto = produto interno (perfil_cliente . perfil_produto)')
print()
print('LIMITACOES:')
print('  Esparsidade alta dificulta o SVD classico')
print('  Alternativas em producao: ALS, two-tower neural, collaborative deep learning')

In [ ]:
# A.3 -- METRICAS DE RECOMENDACAO

def precision_at_k(rec, rel, k):
    return sum(1 for r in rec[:k] if r in rel) / k

def recall_at_k(rec, rel, k):
    hits = sum(1 for r in rec[:k] if r in rel)
    return hits / len(rel) if rel else 0

def ndcg_at_k(rec, rel, k):
    dcg  = sum(1/math.log2(i+2) for i,r in enumerate(rec[:k]) if r in rel)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return dcg/idcg if idcg > 0 else 0

def mrr(rec, rel):
    for i,r in enumerate(rec):
        if r in rel: return 1/(i+1)
    return 0

# Exemplo concreto
recomendados  = [11, 3, 7, 14, 2, 8]   # lista ordenada do modelo
relevantes    = {2, 7, 11}              # o que o cliente realmente vai querer

print('=== METRICAS DE RECOMENDACAO ===')
print()
print(f'Recomendados: {[produtos[i] for i in recomendados]}')
print(f'Relevantes:   {[produtos[i] for i in relevantes]}')
print()

for k in [1, 3, 5]:
    p = precision_at_k(recomendados, relevantes, k)
    r = recall_at_k(recomendados, relevantes, k)
    n = ndcg_at_k(recomendados, relevantes, k)
    print(f'  K={k}:  Precision={p:.4f}  Recall={r:.4f}  NDCG={n:.4f}')

m = mrr(recomendados, relevantes)
pos_1o = int(1/m) if m > 0 else 'nenhuma'
print(f'  MRR: {m:.4f}  (1o relevante na posicao {pos_1o})')
print()
print('INTERPRETACAO:')
print('  Precision@3: dos 3 primeiros recomendados, 66.7% sao relevantes')
print('  Recall@3: capturei 2 dos 3 relevantes nos top-3')
print('  NDCG@3: considera que produto_11 esta na pos 1 (vale mais que pos 3)')
print()
print('QUANDO USAR CADA METRICA NO ITAU:')
print('  Precision@K: campanha cara (ligacao individual) -- qualidade importa')
print('  Recall@K: nao quero perder nenhum interessado')
print('  NDCG@K: app com lista ordenada -- posicao afeta cliques (PRINCIPAL)')
print('  MRR: so o primeiro produto da lista importa (banner unico)')

In [ ]:
# A.4 -- NEXT BEST PRODUCT: PROPENSAO COMO RECOMENDACAO
np.random.seed(42)
n = 2000

df = pd.DataFrame({
    'tempo_conta':  np.random.exponential(4, n),
    'vol_mensal':   np.random.lognormal(7, 1.5, n),
    'n_produtos':   np.random.poisson(3, n) + 1,
    'score_pj':     np.random.normal(650, 80, n).clip(300, 900),
    'sazonalidade': np.random.binomial(1, 0.4, n),
    'ja_credito':   np.random.binomial(1, 0.4, n),
    'inadim_90d':   np.random.binomial(1, 0.08, n),
})

# P(aceita Antecipacao de Recebiveis | features)
prob = 1/(1+np.exp(-(  -3
    + 0.1  * df['tempo_conta']
    + 0.3  * np.log1p(df['vol_mensal'])
    + 0.2  * df['n_produtos']
    + 0.005* (df['score_pj'] - 650)
    + 0.4  * df['sazonalidade']
    + 0.3  * df['ja_credito']
    - 1.5  * df['inadim_90d']
)))
df['aceitou'] = np.random.binomial(1, prob)

print(f'Taxa de aceite: {df["aceitou"].mean()*100:.1f}%')

Xp = df.drop('aceitou', axis=1)
yp = df['aceitou']
Xtr, Xte, ytr, yte = train_test_split(Xp, yp, test_size=0.2, random_state=42)

mod_p = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                       reg_lambda=1.0, verbosity=0, random_state=42)
mod_p.fit(Xtr, ytr)
sc_p  = mod_p.predict_proba(Xte)[:,1]
auc_p = roc_auc_score(yte, sc_p)

# Curva de Lift
df_r = pd.DataFrame({'score': sc_p, 'aceitou': yte.values})
df_r = df_r.sort_values('score', ascending=False).reset_index(drop=True)
taxa_g = df_r['aceitou'].mean()
pcts   = np.arange(5, 105, 5)
lifts  = []
for pct in pcts:
    nk = int(len(df_r)*pct/100)
    tk = df_r['aceitou'][:nk].mean()
    lifts.append(tk/taxa_g if taxa_g > 0 else 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(pcts, lifts, 'b-o', ms=5, lw=2)
axes[0].axhline(1.0, color='gray', linestyle='--', label='Baseline aleatorio')
axes[0].fill_between(pcts, lifts, 1.0, where=np.array(lifts)>1, alpha=0.15)
axes[0].set_xlabel('% da carteira abordada'); axes[0].set_ylabel('Lift')
axes[0].set_title(f'Curva de Lift -- AUC={auc_p:.4f}')
axes[0].legend()

fi = pd.Series(mod_p.feature_importances_, index=Xp.columns)
fi.sort_values(ascending=True).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Feature Importance -- Propensao')
plt.tight_layout(); plt.show()

top10 = lifts[1]
print(f'Lift no top 10%: {top10:.2f}x')
print()
print('PIPELINE NEXT BEST PRODUCT (NBP):')
print('  1. Para cada cliente: calcular P(aceita produto X) via XGBoost')
print('  2. Filtrar produtos ja contratados e inelegiveis')
print('  3. Ordenar por probabilidade: produto_A=0.72, produto_B=0.45, ...')
print('  4. Recomendar top-K: exibir no app ou acionar campanha')
print('  5. Avaliar: Precision@3, Lift, taxa de conversao')
print('  6. Retreinar periodicamente com novos dados de resposta')

In [ ]:
# QUESTOES DO MODULO A
print('=' * 65)
print('QUESTOES DO MODULO A')
print('=' * 65)
print()
print('Q1: Qual a diferenca entre Precision@K e NDCG@K?')
print('    Quando usar cada uma no contexto do Itau?')
print()
print('Q2 (cold start): Novo cliente PJ abriu conta hoje.')
print('    CNAE: 4711 (supermercado), faturamento: R$2M/ano.')
print('    O sistema colaborativo falha (sem historico).')
print('    Descreva sua abordagem para recomendar o 1o produto.')
print()
print('Q3: Como voce conecta um modelo de propensao com recomendacao?')
print('    Descreva o pipeline completo.')
print()
print('-' * 65)
print()
print('GABARITOS:')
print()
print('Q1:')
print('  Precision@K: dos K recomendados, quantos sao relevantes?')
print('  Nao considera a posicao -- pos1 e posK tem mesmo peso.')
print('  Use quando: cada recomendacao tem custo fixo (3 slots no app).')
print()
print('  NDCG@K: posicao 1 vale mais que posicao 5.')
print('  Normalizado: 0 (pessimo) a 1 (ordem perfeita).')
print('  Use quando: lista ordenada e o usuario clica nos primeiros.')
print('  No app do Itau: 1o produto recebe muito mais cliques -> NDCG principal.')
print()
print('Q2: Abordagem cold start para supermercado:')
print('  1. Regras por CNAE 4711: Maquininha, Pix Cobranca, Folha Pagamento')
print('  2. Modelo de propensao com cadastro: CNAE+faturamento+regiao+porte')
print('     Treinado em clientes existentes com perfil similar')
print('  3. Produtos mais populares no segmento como baseline')
print('  4. Apos 30 dias: transicao para modelo colaborativo')
print()
print('Q3: Pipeline NBP:')
print('  Dados -> features (historico, perfil, contexto)')
print('  -> XGBoost por produto -> P(aceita produto X)')
print('  -> Filtrar ja contratados + inelegiveis')
print('  -> Ordenar por probabilidade')
print('  -> Recomendar top-3 no app ou campanha')
print('  -> Registrar feedback -> retreinar periodicamente')

---
# MODULO B -- Experimentacao e Testes A/B
## Prioridade: MUITO ALTA | Vaga menciona 'experimentacao e medicao de impacto'

---

## Por que experimentacao e diferente de analise descritiva?

Analise descritiva: "clientes que usam o app tem NPS 1.2 maior"
Mas esses clientes podem ser naturalmente mais satisfeitos -- confundidor!

Experimento randomizado (A/B test): randomizacao elimina confundidores.
Voce pode afirmar causalidade com confianca.

---

## B.1 Estrutura de um bom teste A/B

1. **Hipotese clara:** nova tela aumenta taxa de ativacao em 7 dias
2. **Metrica primaria:** taxa de ativacao (o que queremos mover)
3. **Metricas de guardrail:** NPS, cancelamento, sessao (NAO podem piorar)
4. **Tamanho de amostra:** calcular ANTES de comecar
5. **Duracao:** definir ANTES (tipicamente 2-4 semanas)
6. **Randomizacao:** A e B estatisticamente equivalentes
7. **Analise:** teste estatistico + IC + tamanho do efeito
8. **Decisao:** por criterio pre-definido, nao so p-valor

---

## B.2 Formula do tamanho de amostra

Para duas proporcoes:
```
n = (z_alpha/2 + z_beta)^2 * (p1*(1-p1) + p2*(1-p2)) / delta^2
```
- z_alpha/2 = 1.96 (alpha=5%, bicaudal)
- z_beta = 0.84 (poder=80%)
- delta = MDE (minimo efeito detectavel)

MDE: qual o menor efeito que muda a decisao de negocio?
Deve vir do negocio, nao de calculos estatisticos.

---

## B.3 O problema do Peeking

Checar o p-valor todo dia e parar quando p < 0.05
infla a taxa de falso positivo de 5% para 20-30%+.

Analogia: voce apostou em 10 lancamentos de moeda.
Apos 3 lancamentos (3 caras), nao pode parar e dizer que venceu.
Voce definiu 10 lancamentos -- isso e o contrato.

Solucao: definir duracao ANTES. So analisar ao final.

---

## B.4 Diferencas em Diferencas (DiD)

Quando A/B test nao e possivel (politica afeta toda uma regiao).

Formula:
```
DiD = (tratado_depois - tratado_antes) - (controle_depois - controle_antes)
```

Premissa critica: Tendencias Paralelas.
Sem o tratamento, tratado e controle teriam evoluido identicamente.
Verificar: plotar as series antes do tratamento -- devem ser paralelas.


In [ ]:
# B.1 -- CALCULO DE TAMANHO DE AMOSTRA
from scipy.stats import norm

def n_amostra(p_base, mde, alpha=0.05, poder=0.80):
    # p_base: taxa baseline no controle
    # mde: minimo efeito detectavel (diferenca absoluta)
    z_a = norm.ppf(1 - alpha/2)    # z critico para alpha bicaudal
    z_b = norm.ppf(poder)           # z critico para o poder desejado
    p2  = p_base + mde              # taxa esperada no tratamento
    n   = (z_a + z_b)**2 * (p_base*(1-p_base) + p2*(1-p2)) / (mde**2)
    return math.ceil(n)

print('=== TAMANHO DE AMOSTRA ===')
print()
print('Cenario: nova tela de onboarding no app Itau PJ')
print('Baseline: 8.0% de ativacao em 7 dias')
print('MDE: detectar melhora de 2pp (de 8% para 10%)')
print()

p_b = 0.08
mde_ex = 0.02
n_nec = n_amostra(p_b, mde_ex)
print(f'  n necessario por grupo: {n_nec:,}')
print(f'  Total (A + B):          {n_nec*2:,} clientes')
print()

print('TABELA: n por grupo (alpha=5%, baseline=8%)')
print()
print(f'{"MDE":>8} {"Poder 70%":>12} {"Poder 80%":>12} {"Poder 90%":>12}')
print('-' * 48)
for mde_pct in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    mv = mde_pct / 100
    n70 = n_amostra(p_b, mv, poder=0.70)
    n80 = n_amostra(p_b, mv, poder=0.80)
    n90 = n_amostra(p_b, mv, poder=0.90)
    print(f'{mde_pct:>7.1f}%  {n70:>12,} {n80:>12,} {n90:>12,}')

print()
print('INTERPRETACAO:')
print('  MDE menor = detectar efeitos pequenos = MUITO mais amostra')
print('  Poder maior = mais certeza = mais amostra')
print('  MDE deve vir do negocio: qual o menor efeito que muda a decisao?')
print()

# Visualizando tradeoff
mdes = np.arange(0.005, 0.060, 0.002)
ns80 = [n_amostra(p_b, m, poder=0.80) for m in mdes]
ns90 = [n_amostra(p_b, m, poder=0.90) for m in mdes]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mdes*100, ns80, 'b-', lw=2, label='Poder=80%')
ax.plot(mdes*100, ns90, 'r-', lw=2, label='Poder=90%')
ax.axvline(mde_ex*100, color='green', linestyle='--', lw=1.5,
           label=f'MDE={mde_ex*100:.0f}%')
ax.set_xlabel('MDE (pp)'); ax.set_ylabel('N por grupo')
ax.set_title('Tradeoff: MDE vs Tamanho de Amostra'); ax.legend()
ax.set_yscale('log')
plt.tight_layout(); plt.show()

In [ ]:
# B.2 -- EXECUTANDO E ANALISANDO UM TESTE A/B
from scipy.stats import chi2_contingency
np.random.seed(42)

n_grupo      = 5000
p_controle   = 0.082    # taxa de ativacao atual
p_tratamento = 0.098    # nova tela (efeito real = +1.6pp)

ctrl = np.random.binomial(1, p_controle,   n_grupo)
trat = np.random.binomial(1, p_tratamento, n_grupo)

tc = ctrl.mean(); tt = trat.mean(); dif = tt - tc
se = np.sqrt(tc*(1-tc)/n_grupo + tt*(1-tt)/n_grupo)
ic = (dif - 1.96*se, dif + 1.96*se)

tab = np.array([[ctrl.sum(), n_grupo-ctrl.sum()],
                [trat.sum(), n_grupo-trat.sum()]])
chi2, pval, _, _ = chi2_contingency(tab)

print('=' * 55)
print('RESULTADO DO TESTE A/B')
print('=' * 55)
print()
print(f'CONTROLE:    N={n_grupo:,}  Ativacoes={ctrl.sum():,}  Taxa={tc*100:.2f}%')
print(f'TRATAMENTO:  N={n_grupo:,}  Ativacoes={trat.sum():,}  Taxa={tt*100:.2f}%')
print()
print(f'Diferenca: {dif*100:+.2f}pp')
print(f'IC 95%:    [{ic[0]*100:.2f}pp, {ic[1]*100:.2f}pp]')
print(f'Chi2:      {chi2:.4f}')
print(f'P-valor:   {pval:.4f}')
print()

if pval < 0.05:
    print('p < 0.05 -> RESULTADO SIGNIFICATIVO')
    print('Evidencia de que a nova tela melhora a ativacao')
    if ic[0] > 0:
        print('IC 95% completamente acima de 0 -> resultado robusto')
else:
    print('p >= 0.05 -> SEM EVIDENCIA de diferenca')

print()
print('COMO REPORTAR PARA O GESTOR:')
print('  NAO: "p=0.02, resultado significativo"')
print('  SIM: "A nova tela aumentou ativacao em +1.6pp (IC 95%: [+0.9, +2.3]).')
print('       Com 80k novos clientes/mes, ~1.280 ativacoes extras por mes.')
print('       Receita incremental estimada de R$X/mes."')

In [ ]:
# B.3 -- PROBLEMA DO PEEKING
from scipy.stats import chi2_contingency
np.random.seed(42)

n_sims   = 2000
p_real   = 0.10     # MESMO efeito nos dois grupos (sem diferenca real)
n_max    = 3000     # tamanho pre-definido

fp_peek  = 0        # falsos positivos com peeking
fp_ok    = 0        # falsos positivos sem peeking

for _ in range(n_sims):
    c_acc, t_acc = [], []
    achou = False
    for t in range(20, n_max+1, 20):
        c_acc.extend(np.random.binomial(1, p_real, 20).tolist())
        t_acc.extend(np.random.binomial(1, p_real, 20).tolist())
        if t < n_max and not achou and len(c_acc) >= 40:
            nc, nt = np.array(c_acc), np.array(t_acc)
            tab = np.array([[nc.sum(), len(nc)-nc.sum()],
                            [nt.sum(), len(nt)-nt.sum()]])
            try:
                _, p, _, _ = chi2_contingency(tab)
                if p < 0.05:
                    fp_peek += 1; achou = True
            except: pass
    nc, nt = np.array(c_acc[:n_max]), np.array(t_acc[:n_max])
    tab = np.array([[nc.sum(), len(nc)-nc.sum()],
                    [nt.sum(), len(nt)-nt.sum()]])
    try:
        _, p_f, _, _ = chi2_contingency(tab)
        if p_f < 0.05: fp_ok += 1
    except: pass

print('=== PROBLEMA DO PEEKING ===')
print()
print(f'{n_sims} experimentos SEM efeito real (mesma taxa nos dois grupos)')
print()
print(f'  Taxa falso positivo COM peeking:  {fp_peek/n_sims*100:.1f}%  <- deveria ser 5%!')
print(f'  Taxa falso positivo SEM peeking:  {fp_ok/n_sims*100:.1f}%  <- correto')
print()
print('CONCLUSAO: peeking infla o falso positivo de 5% para 20-30%+')
print('Voce encontra "resultado" mesmo quando nao existe.')
print()
print('COMO EVITAR:')
print('  1. Calcule n necessario ANTES de comecar')
print('  2. Defina a duracao ANTES (ex: 4 semanas)')
print('  3. So analise ao final -- resistir a tentacao')
print('  4. Para monitoramento continuo: usar Sequential Testing (SPRT)')

In [ ]:
# B.4 -- DIFERENCAS EM DIFERENCAS (DiD)
np.random.seed(42)
meses = np.arange(12)

# Politica implementada em SP no mes 6
sp_antes  = 7.0 + 0.1*meses[:6] + np.random.normal(0, 0.2, 6)
sp_depois = 7.6 + 0.1*meses[6:] + np.random.normal(0, 0.2, 6)  # +0.6 por efeito
sp_nps    = np.concatenate([sp_antes, sp_depois])
rj_nps    = 6.8 + 0.1*meses + np.random.normal(0, 0.2, 12)    # sem politica

sA, sD = sp_nps[:6].mean(), sp_nps[6:].mean()
rA, rD = rj_nps[:6].mean(), rj_nps[6:].mean()
did    = (sD - sA) - (rD - rA)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(meses[:6], sp_nps[:6], 'b-o', ms=6, label='SP (tratado) - antes')
axes[0].plot(meses[6:], sp_nps[6:], 'b-s', ms=6, label='SP (tratado) - depois')
axes[0].plot(meses,     rj_nps,     'r-o', ms=6, label='RJ+MG (controle)')
axes[0].axvline(5.5, color='gray', linestyle='--', lw=2, label='Politica')
axes[0].set_title('Diferencas em Diferencas\nTendencias paralelas antes do tratamento')
axes[0].legend(fontsize=8)

grupos = ['Ctrl\nAntes', 'Ctrl\nDepois', 'Trat\nAntes', 'Trat\nDepois']
vals   = [rA, rD, sA, sD]
cores  = ['lightcoral','coral','lightblue','steelblue']
axes[1].bar(grupos, vals, color=cores, edgecolor='black', width=0.5)
axes[1].set_title(f'Decomposicao\nDelta_SP={sD-sA:.2f} | Delta_RJ={rD-rA:.2f} | DiD={did:.2f}')
plt.tight_layout(); plt.show()

print(f'DiD = ({sD:.2f}-{sA:.2f}) - ({rD:.2f}-{rA:.2f}) = {did:.2f} pontos de NPS')
print()
print('PREMISSA CRITICA: Tendencias Paralelas')
print('  Sem a politica, SP e RJ teriam evoluido da mesma forma.')
print('  Verificar: as series antes do tratamento sao paralelas?')
print('  Se SP ja crescia mais que RJ antes: DiD e enviesado.')
print()
print('QUANDO USAR DiD:')
print('  - Politica de credito que afeta toda uma regiao')
print('  - Mudanca de produto so possivel em certas agencias')
print('  - Restricao legal de randomizar (regulacao de credito)')

In [ ]:
# QUESTOES DO MODULO B
print('=' * 65)
print('QUESTOES DO MODULO B')
print('=' * 65)
print()
print('Q1 (calculo): Taxa de clique baseline = 5.0%.')
print('    Voce quer detectar melhora de 1pp (5%->6%).')
print('    Alpha=5%, poder=80%. Qual o n por grupo?')
print()
print('Q2 (peeking): Gestor quer parar o teste apos 3 dias.')
print('    N atual: 1.200/grupo. N necessario: 4.500/grupo. P-valor: 0.03.')
print('    O que voce diz?')
print()
print('Q3 (guardrail): Metrica primaria melhorou +2.1pp (p=0.02).')
print('    Metrica de guardrail (tempo na sessao) caiu 15%.')
print('    Qual sua recomendacao?')
print()
print('-' * 65)
print()

n_q1 = n_amostra(0.05, 0.01)
print('GABARITOS:')
print()
print(f'Q1: n = {n_q1:,} por grupo  (total: {n_q1*2:,})')
print('    z_alpha=1.96, z_beta=0.84, p1=0.05, p2=0.06, delta=0.01')
print()
print('Q2: NAO parar. Explicar ao gestor:')
print('  "N=1.200 e 27% do necessario (4.500)."')
print('  "Parar com p=0.03 agora e peeking."')
print('  "A taxa real de falso positivo pode ser 20%+, muito acima do 5%."')
print('  "Precisamos continuar ate 4.500/grupo para confiar no resultado."')
print()
print('Q3: NAO lanccar. Guardrail violado.')
print('  Metrica primaria melhorou, mas sessao caiu 15% -- sinal grave.')
print('  Possivel: nova tela e mais agressiva, converte rapido mas desengaja.')
print('  Guardrails existem para evitar otimizar a metrica errada.')
print('  Acao: investigar por que a sessao caiu, revisar o design, re-testar.')

---
# MODULO C -- AWS e SageMaker
## Prioridade: ALTA | Entrevista e trabalho real

---

## Pipeline tipico no Itau com AWS

```
Dados transacionais
   -> S3 (raw data lake)
   -> Glue (catalogo + ETL)
   -> Athena (SQL exploration + features)
   -> S3 (features processadas)
   -> SageMaker Training Job (treino)
   -> S3 (modelo serializado)
   -> MLflow Model Registry (versionamento)
   -> SageMaker Endpoint (real-time) OU Batch Transform (batch)
   -> Model Monitor (drift, metricas)
   -> QuickSight (dashboard de performance)
```

---

## Servicos essenciais

**S3:** armazenamento. O "HD" da cloud. Dados, modelos, features, logs.

**Athena:** SQL diretamente no S3 sem ETL. Cobra por dado escaneado.
Sempre use filtros de data para nao escanear tudo.

**SageMaker:** plataforma completa de ML.
- Studio: notebook gerenciado
- Training Jobs: treino em instancias dedicadas (CPU/GPU)
- Endpoints: servir modelos real-time
- Batch Transform: scoring de grandes volumes offline
- Model Monitor: deteccao de drift automatica
- Feature Store: features centralizadas e reutilizaveis

---

## Endpoint vs Batch Transform

| | Endpoint Real-time | Batch Transform |
|--|-------------------|-----------------|
| Latencia | < 100ms | Horas |
| Volume | 1 requisicao | Milhoes |
| Custo | Por requisicao (alto) | Por instancia (barato) |
| Uso Itau | Fraude em Pix | Propensao semanal |

---

## PSI: monitoramento de drift

PSI < 0.10: mudanca insignificante -- modelo valido
PSI 0.10-0.25: investigar
PSI > 0.25: RETREINAR -- modelo invalido


In [ ]:
# C.1 -- PIPELINE AWS (simulado localmente)
print('=== PIPELINE ML NO ITAU COM AWS ===')
print()
print('ETAPA 1: Dados no S3')
print('  s3://itau-datalake-pj/raw/transacoes/2024/')
print('  s3://itau-datalake-pj/features/propensao_antecipacao/')
print()
print('ETAPA 2: Feature Engineering via Athena (SQL no S3)')
query = [
    'SELECT',
    '  c.cnpj, c.cnae, c.porte,',
    '  AVG(t.valor)  AS vol_medio_90d,',
    '  COUNT(t.id)   AS n_trans_90d,',
    '  c.score_pj, c.inadim_90d',
    'FROM clientes c',
    'JOIN transacoes t ON c.cnpj = t.cnpj',
    'WHERE t.data >= DATE_ADD(day, -90, CURRENT_DATE)',
    'GROUP BY c.cnpj, c.cnae, c.porte, c.score_pj, c.inadim_90d'
]
for linha in query:
    print(f'  {linha}')
print()
print('ETAPA 3: SageMaker Training Job')
treino = [
    'from sagemaker.sklearn import SKLearn',
    'estimator = SKLearn(',
    '    entry_point="train.py",',
    '    role=role,',
    '    instance_type="ml.m5.large",',
    '    hyperparameters={"n-estimators": 200, "max-depth": 4}',
    ')',
    'estimator.fit({"train": "s3://bucket/features/treino/"})'
]
for linha in treino:
    print(f'  {linha}')
print()
print('ETAPA 4: Deploy como Endpoint (real-time -- fraude)')
deploy = [
    'predictor = estimator.deploy(',
    '    initial_instance_count=1,',
    '    instance_type="ml.t2.medium",',
    '    endpoint_name="propensao-antecipacao-v3"',
    ')',
    'resultado = predictor.predict([[features_cnpj]])',
    'p_aceite  = resultado[0]'
]
for linha in deploy:
    print(f'  {linha}')
print()
print('ETAPA 5: Batch Transform (propensao semanal -- 2M de CNPJs)')
batch = [
    'transformer = estimator.transformer(',
    '    instance_count=1,',
    '    instance_type="ml.m5.xlarge",',
    '    output_path="s3://bucket/scores/propensao/"',
    ')',
    'transformer.transform(',
    '    data="s3://bucket/features/carteira/",',
    '    content_type="text/csv",',
    '    split_type="Line"',
    ')'
]
for linha in batch:
    print(f'  {linha}')
print()
print('ENDPOINT vs BATCH:')
print('  Endpoint: decisao em <100ms (fraude, scoring em tempo real)')
print('  Batch: scoring de milhoes offline, barato, pode esperar horas')
print('  Regra: se precisa de resultado imediato -> endpoint')
print('         se pode esperar -> batch (muito mais economico)')

In [ ]:
# C.2 -- MONITORAMENTO DE DRIFT COM PSI
def calcular_psi(ref, atual, n_bins=10):
    bins = np.percentile(ref, np.linspace(0, 100, n_bins+1))
    bins[0] = -np.inf; bins[-1] = np.inf
    fr = np.histogram(ref,   bins=bins)[0] / len(ref)
    fa = np.histogram(atual, bins=bins)[0] / len(atual)
    fr = np.where(fr == 0, 0.0001, fr)
    fa = np.where(fa == 0, 0.0001, fa)
    return np.sum((fa - fr) * np.log(fa / fr))

np.random.seed(42)
n_ref, n_prod = 5000, 2000

# Simulando cenario de crise economica: distribuicoes mudaram
features_psi = {
    'vol_mensal':  (np.random.lognormal(7,1.5,n_ref), np.random.lognormal(7.8,1.5,n_prod)),
    'n_produtos':  (np.random.poisson(3,n_ref),        np.random.poisson(3.1,n_prod)),
    'score_pj':    (np.random.normal(650,80,n_ref),    np.random.normal(615,90,n_prod)),
    'inadim_90d':  (np.random.binomial(1,.08,n_ref),   np.random.binomial(1,.19,n_prod)),
}

print('=== MONITORAMENTO DE DRIFT: PSI ===')
print()
print(f'{"Feature":<15} {"PSI":>8} {"Status":<15} Acao')
print('-' * 60)

for feat, (ref, atual) in features_psi.items():
    psi = calcular_psi(ref, atual)
    if psi < 0.10:
        status, acao = 'OK', 'Nenhuma'
    elif psi < 0.25:
        status, acao = 'ATENCAO', 'Investigar'
    else:
        status, acao = 'RETREINAR!', 'URGENTE: modelo invalido'
    print(f'{feat:<15} {psi:>8.4f} {status:<15} {acao}')

print()
print('INTERPRETACAO:')
print('  inadim_90d: PSI alto = taxa foi de 8% para 19% (crise!)')
print('  Modelo treinado em distribuicao muito diferente da atual')
print('  Retreinar urgente com dados dos ultimos 6 meses')
print()
print('NO SAGEMAKER MODEL MONITOR:')
monitor_code = [
    'model_monitor = predictor.run_model_monitor_schedule(',
    '    monitor_schedule_name="propensao-monitor-diario",',
    '    statistics=baseline_statistics,    # distribuicao do treino',
    '    constraints=baseline_constraints,  # thresholds de PSI',
    '    schedule_cron_expression="cron(0 * * * ? *)"',
    ')'
]
for linha in monitor_code:
    print(f'  {linha}')

In [ ]:
# QUESTOES DO MODULO C
print('=' * 65)
print('QUESTOES DO MODULO C')
print('=' * 65)
print()
print('Q1: Voce vai construir deteccao de fraude em Pix (<200ms).')
print('    Descreva a arquitetura AWS que voce usaria.')
print()
print('Q2: Modelo de propensao: campanha semanal, 2M CNPJs.')
print('    Endpoint real-time ou Batch Transform? Por que?')
print()
print('Q3: PSI do score_pj = 0.28. O que voce faz?')
print()
print('-' * 65)
print()
print('GABARITOS:')
print()
print('Q1: Arquitetura fraude real-time:')
print('  Transacao -> Kinesis (stream em tempo real)')
print('  -> Feature Store (features pre-computadas do cliente <10ms)')
print('  -> SageMaker Endpoint (ml.c5.large com auto-scaling)')
print('  -> Decisao em <50ms + monitoramento CloudWatch')
print('  -> Model Monitor para detectar drift')
print()
print('Q2: Batch Transform.')
print('  2M CNPJs: volume alto, sem urgencia (pode esperar 1-2h).')
print('  Batch cobra so pelo tempo da instancia durante o job.')
print('  Resultado: arquivo CSV no S3 que a equipe comercial usa.')
print('  Endpoint mantido ativo 24h seria muito caro sem necessidade.')
print()
print('Q3: PSI=0.28 > 0.25 = mudanca SEVERA.')
print('  1. Alertar time de negocio e dados imediatamente')
print('  2. Investigar CAUSA: mudanca regulatoria? crise? sazonalidade?')
print('  3. Verificar AUC atual: se caiu >5pp -> retreinar urgente')
print('  4. Coletar dados recentes (3-6 meses) para retreino')
print('  5. Retreinar, validar em holdout temporal, promover via Registry')
print('  6. Documentar: quando, causa, acao tomada (auditoria BACEN)')

---
# MODULO D -- Prova Estimada 2026
## 40 Questoes com gabarito completo

**Como simular a prova:**
1. Feche todos os gabaritos
2. Cronometro: 90 minutos para as 40 questoes
3. Questoes praticas: rode o codigo, anote o resultado
4. So veja o gabarito depois

**Distribuicao estimada:**
- 15 questoes conceituais (0.2pt = 3.0pts)
- 10 questoes praticas com codigo (0.5pt = 5.0pts)
- 10 questoes tematicas da vaga (0.2pt = 2.0pts)
- 5 questoes situacionais (0.5pt = 2.5pts)

**Total: 12.5 pontos**


In [ ]:
# SECAO 1: QUESTOES CONCEITUAIS (15 questoes)
conceituais = [
    (1,  'P-valor = 0.03 significa:',
         ['(a) 97% de chance do efeito ser real',
          '(b) 3% de chance de H0 ser verdadeira',
          '(c) Se H0 fosse verdade, resultado ocorreria 3% das vezes',
          '(d) Significativo com alpha=0.01'],
         '(c)', 'Definicao exata de p-valor. (a) e (b) sao erros classicos.'),

    (2,  'R^2 ao adicionar variavel irrelevante:',
         ['(a) Diminui',
          '(b) Permanece igual',
          '(c) Nunca diminui',
          '(d) Pode aumentar ou diminuir'],
         '(c)', 'R^2 NUNCA diminui. R^2 ajustado PODE diminuir.'),

    (3,  'Qual funcao de ativacao NAO pode gerar -0.001?',
         ['(a) Tanh',
          '(b) Leaky ReLU',
          '(c) ReLU',
          '(d) Linear'],
         '(c)', 'ReLU=max(0,x) sempre >=0. Tanh e LeakyReLU podem ser negativos.'),

    (4,  'lambda->infinito em Ridge: coeficientes vao para?',
         ['(a) Infinito',
          '(b) Zero',
          '(c) OLS',
          '(d) 1'],
         '(b)', 'Lambda grande encolhe coefs em direcao a zero. Nunca para infinito.'),

    (5,  'C=0.001 em LogisticRegression significa:',
         ['(a) Regularizacao fraca',
          '(b) Regularizacao forte (alpha=1000)',
          '(c) Sem regularizacao',
          '(d) Ridge com alpha=0.001'],
         '(b)', 'C=1/alpha. C=0.001 -> alpha=1000 -> regularizacao MUITO forte.'),

    (6,  'Padrao de CV da prova Itau:',
         ['(a) TimeSeriesSplit(5)',
          '(b) KFold(5, shuffle=True)',
          '(c) KFold(5, shuffle=False)',
          '(d) StratifiedKFold(5)'],
         '(c)', 'Confirmado empiricamente com Q10 e Q11 da prova de 2019.'),

    (7,  'neg_log_loss retorna -0.45. O Log Loss real e:',
         ['(a) -0.45', '(b) 0.45', '(c) 0.55', '(d) -0.55'],
         '(b)', 'Sempre multiplicar neg_* por -1.'),

    (8,  'Arvore sem poda, entropy, KFold10. Log Loss treino:',
         ['(a) ~0.5', '(b) ~1.0', '(c) ~0.0', '(d) ~3.0'],
         '(c)', 'Folhas puras: P=1.0 no treino, -log(1)=0. Q20 prova 2019.'),

    (9,  'Random Forest: n_estimators de 100 para 500 causa:',
         ['(a) Overfitting',
          '(b) Underfitting',
          '(c) Maior estabilidade, sem mais overfitting',
          '(d) Reduz AUC'],
         '(c)', 'RF: mais arvores = mais estavel. Overfit vem de max_depth, nao n_est.'),

    (10, 'Average linkage calcula distancia entre clusters como:',
         ['(a) Distancia entre centroides',
          '(b) Distancia minima entre pontos',
          '(c) Media par-a-par',
          '(d) Distancia maxima'],
         '(c)', 'Average = media par-a-par. NAO e centroide. Q31 prova 2019.'),

    (11, 'Cold start em recomendacao ocorre quando:',
         ['(a) Matriz densa',
          '(b) Novo usuario sem historico',
          '(c) Muitos fatores latentes',
          '(d) Dois usuarios identicos'],
         '(b)', 'Sem historico, filtragem colaborativa falha.'),

    (12, 'Peeking em A/B test causa:',
         ['(a) Reducao do poder',
          '(b) Inflacao da taxa de falso positivo',
          '(c) Reduz tamanho de amostra',
          '(d) Aumenta o IC'],
         '(b)', 'Checagens multiplas inflam erro tipo 1 de 5% para 20-30%+.'),

    (13, 'NDCG@K difere de Precision@K pois:',
         ['(a) Usa apenas o primeiro relevante',
          '(b) Penaliza relevantes em posicoes inferiores',
          '(c) Nao precisa de ground truth',
          '(d) E igual a Recall@K'],
         '(b)', 'NDCG desconta por posicao: pos1 vale mais que pos5.'),

    (14, 'PSI de 0.31 indica:',
         ['(a) Mudanca insignificante',
          '(b) Mudanca moderada',
          '(c) Mudanca severa -- retreinar',
          '(d) Erro no calculo'],
         '(c)', 'PSI > 0.25 = severo. Retreinar o modelo.'),

    (15, 'MAPE com y_true contendo zero:',
         ['(a) MAPE = 0',
          '(b) MAPE = infinito',
          '(c) MAPE e INDEFINIDO',
          '(d) MAPE ignora o zero'],
         '(c)', 'Divisao por |y_true|=0. Nunca use MAPE com zeros. Q13 prova 2019.'),
]

print('=' * 70)
print('SECAO 1 -- CONCEITUAIS (15 questoes, 0.2pt cada)')
print('Responda antes de ver o gabarito!')
print('=' * 70)
print()

for num, perg, alts, gab, exp in conceituais:
    print(f'Q{num:02d}: {perg}')
    for a in alts: print(f'  {a}')
    print()

print()
print('=' * 70)
print('GABARITOS -- SECAO 1')
print('=' * 70)
for num, perg, alts, gab, exp in conceituais:
    print(f'  Q{num:02d}: {gab} -- {exp}')

In [ ]:
# SECAO 2: QUESTOES PRATICAS (10 questoes, 0.5pt cada)
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np, pandas as pd, os, math

kf5  = KFold(n_splits=5,  shuffle=False)
kf10 = KFold(n_splits=10, shuffle=False)

def cv_neg(modelo, X, y, kf, scoring):
    r = cross_validate(modelo, X, y, cv=kf, scoring=scoring, return_train_score=True)
    tr = -r['train_score'].mean() if 'neg_' in scoring else r['train_score'].mean()
    va = -r['test_score'].mean()  if 'neg_' in scoring else r['test_score'].mean()
    return round(tr, 4), round(va, 4)

print('=' * 70)
print('SECAO 2 -- PRATICAS (10 questoes, 0.5pt cada)')
print('=' * 70)
print()

# Q16 -- ElasticNet
if os.path.exists('regressao_Q1.csv'):
    df = pd.read_csv('regressao_Q1.csv'); X, y = df.drop('target',axis=1), df['target']
    tr, va = cv_neg(ElasticNet(alpha=1.0, l1_ratio=0.01), X, y, kf5, 'neg_mean_squared_error')
    print(f'Q16 -- ElasticNet (alpha=1.0, l1_ratio=0.01, KFold5):')
    print(f'  Treino MSE={tr} | Val MSE={va}  (gabarito: ~0.2686 e ~0.2693)')
else:
    print('Q16: regressao_Q1.csv nao encontrado')

# Q17 -- SVR
if os.path.exists('regressao_Q2.csv'):
    df = pd.read_csv('regressao_Q2.csv'); X, y = df.drop('target',axis=1), df['target']
    tr, va = cv_neg(SVR(kernel='linear', C=0.001), X, y, kf5, 'neg_mean_squared_error')
    print(f'Q17 -- SVR (linear, C=0.001, KFold5):')
    print(f'  Treino MSE={tr} | Val MSE={va}  (gabarito: ~20199 e ~20207)')
else:
    print('Q17: regressao_Q2.csv nao encontrado')

# Q18 -- Arvore Log Loss
if os.path.exists('classificacao_Q1.csv'):
    df = pd.read_csv('classificacao_Q1.csv'); X, y = df.drop('target',axis=1), df['target']
    tr, va = cv_neg(DecisionTreeClassifier(criterion='entropy'), X, y, kf10, 'neg_log_loss')
    print(f'Q18 -- Arvore sem poda (entropy, KFold10):')
    print(f'  Treino LL={tr} | Val LL={va}  (treino ~0 = overfitting total)')
else:
    print('Q18: classificacao_Q1.csv nao encontrado')

# Q19 -- Logistica
if os.path.exists('classificacao_Q2.csv'):
    df = pd.read_csv('classificacao_Q2.csv'); X, y = df.drop('target',axis=1), df['target']
    tr, va = cv_neg(LogisticRegression(C=0.1, max_iter=1000), X, y, kf10, 'roc_auc')
    print(f'Q19 -- Logistica (C=0.1, L2, KFold10):')
    print(f'  Treino AUC={tr} | Val AUC={va}  (regularizado: treino~val)')
else:
    print('Q19: classificacao_Q2.csv nao encontrado')

# Q20 -- Ridge
if os.path.exists('regressao_Q1.csv'):
    df = pd.read_csv('regressao_Q1.csv'); X, y = df.drop('target',axis=1), df['target']
    tr, va = cv_neg(Ridge(alpha=5.0), X, y, kf5, 'neg_mean_squared_error')
    print(f'Q20 -- Ridge (alpha=5.0, KFold5): Treino={tr} | Val={va}')
else:
    print('Q20: regressao_Q1.csv nao encontrado')

# Q21 -- SVM RBF
if os.path.exists('classificacao_Q2.csv'):
    df = pd.read_csv('classificacao_Q2.csv'); X, y = df.drop('target',axis=1), df['target']
    pipe = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=1.0, gamma=0.1, probability=True))])
    tr, va = cv_neg(pipe, X, y, kf5, 'roc_auc')
    print(f'Q21 -- SVM rbf (C=1.0, gamma=0.1, KFold5, normalizado):')
    print(f'  Treino AUC={tr} | Val AUC={va}')
else:
    print('Q21: classificacao_Q2.csv nao encontrado')

# Q22 -- Profundidade maxima
n22 = 10_000_000
prof22 = math.ceil(math.log2(n22))
print(f'Q22 -- Profundidade max (n=10M): ceil(log2({n22:,})) = ceil({math.log2(n22):.2f}) = {prof22}')

# Q23 -- Metricas de classificacao
print('Q23 -- Metricas (TP=80, TN=720, FP=30, FN=170):')
TP,TN,FP,FN = 80,720,30,170
total = TP+TN+FP+FN
acc  = (TP+TN)/total
prec = TP/(TP+FP)
rec  = TP/(TP+FN)
f1   = 2*prec*rec/(prec+rec)
print(f'  Acuracia={acc:.4f} | Precision={prec:.4f} | Recall={rec:.4f} | F1={f1:.4f}')

# Q24 -- Precision@K e NDCG@K
rec24 = ['A','B','C','D','E']
rel24 = {'A','C','E'}
k24   = 3
hits24  = [1 if r in rel24 else 0 for r in rec24[:k24]]
prec24  = sum(hits24)/k24
dcg24   = sum(h/math.log2(i+2) for i,h in enumerate(hits24))
idcg24  = sum(1/math.log2(i+2) for i in range(min(len(rel24),k24)))
ndcg24  = dcg24/idcg24
print(f'Q24 -- Precision@3={prec24:.4f} | NDCG@3={ndcg24:.4f}')
print('       Recomendados:[A,B,C,D,E] | Relevantes:{A,C,E}')

# Q25 -- Tamanho de amostra
from scipy.stats import norm as norm_dist
p25, m25 = 0.05, 0.01
za25 = norm_dist.ppf(1-0.05/2); zb25 = norm_dist.ppf(0.80)
n25  = math.ceil((za25+zb25)**2*(p25*(1-p25)+(p25+m25)*(1-(p25+m25)))/m25**2)
print(f'Q25 -- Tamanho de amostra (baseline=5%, MDE=1pp, alpha=5%, poder=80%):')
print(f'       n = {n25:,} por grupo  (total: {n25*2:,})')

In [ ]:
# SECAO 3: QUESTOES SITUACIONAIS (5 questoes, 0.5pt cada)
print('=' * 70)
print('SECAO 3 -- SITUACIONAIS (5 questoes, 0.5pt cada)')
print('=' * 70)
print()
print('Q26 -- OVERFITTING vs LEAKAGE:')
print('  XGBoost de inadimplencia: AUC treino=0.96, AUC teste=0.72.')
print('  Colega: overfitting. Voce: suspeita de leakage.')
print('  (a) Como distinguir os dois?')
print('  (b) Qual feature investigar primeiro?')
print('  (c) Como corrigir se for leakage?')
print()
print('Q27 -- TESTE A/B:')
print('  Gestor quer parar o teste: 2 dias, n=800/grupo, p=0.03.')
print('  N necessario calculado: 3.200/grupo.')
print('  (a) O que voce diz? (b) Quais guardrails monitorar?')
print()
print('Q28 -- COLD START:')
print('  Supermercado abriu conta no Itau hoje (sem historico).')
print('  Descreva sua abordagem para recomendar o 1o produto.')
print()
print('Q29 -- DRIFT:')
print('  PSI de vol_mensal = 0.35. AUC caiu de 0.82 para 0.78.')
print('  (a) O que cada indicador sugere?')
print('  (b) Voce retreinaria imediatamente?')
print()
print('Q30 -- COMUNICACAO EXECUTIVA:')
print('  Modelo: AUC=0.81, Lift top10%=2.6x, Precision@3=0.47.')
print('  Explique em 3 frases para o diretor comercial o impacto esperado.')
print()
print('-' * 70)
print()
print('GABARITOS -- SECAO 3:')
print()
print('Q26:')
print('  (a) Overfitting: degrada gradualmente em dados novos.')
print('      Leakage: colapsa totalmente em producao (AUC volta a 0.5-0.6).')
print('      Teste: avaliar em janela temporal futura completamente separada.')
print()
print('  (b) Feature com importancia inesperadamente alta.')
print('      Ex: status_cobranca, data_vencimento, valor_atraso_atual.')
print('      Qualquer feature que so existe DEPOIS que o cliente inadimpliu.')
print()
print('  (c) Remover features com leakage. Recriar dataset com split temporal.')
print('      Treinar em Jan-Jun. Testar em Jul-Dez. Nunca usar futuro no treino.')
print()
print('Q27:')
print('  (a) "N=800 e 25% do necessario (3.200)."')
print('      "Parar com p=0.03 e peeking: falso positivo pode ser 20%+."')
print('      "Precisamos continuar ate 3.200/grupo para confiar."')
print()
print('  (b) Guardrails para teste de onboarding:')
print('      - Taxa de abandono do app (nao pode subir)')
print('      - NPS no feedback imediato (nao pode cair)')
print('      - Tempo medio de sessao (nao pode cair)')
print()
print('Q28: Cold start para supermercado:')
print('  1. Regras por CNAE 4711: Maquininha, Pix Cobranca, Folha Pagamento')
print('  2. Modelo de propensao com cadastro (CNAE+faturamento+regiao+porte)')
print('     Treinado em clientes existentes com perfil similar')
print('  3. Apos 30 dias: transicao para modelo colaborativo')
print()
print('Q29:')
print('  (a) PSI=0.35: distribuicao do volume MUITO diferente do treino.')
print('      AUC caindo 4pp: degradacao real, mas ainda dentro de margem.')
print()
print('  (b) Nao necessariamente imediato. Perguntas antes:')
print('      - Tendencia ou sazonalidade? - Impacto no negocio?')
print('      - Ha dados recentes suficientes para retreinar?')
print('      Se persistente e AUC continuando a cair: retreinar urgente.')
print()
print('Q30: Traducao para o diretor:')
print('  "Ao abordar os 10% mais propensos, a taxa de aceite e 2.6x maior')
print('   que uma campanha aleatoria -- reduzindo custo por contrato em 62%."')
print('  "Dos 3 produtos recomendados, em media 1.4 sao aceitos (Precision@3=0.47),')
print('   contra 0.5 de uma recomendacao aleatoria."')
print('  "Com 500k clientes elegiveis e custo de R$15/abordagem,')
print('   o modelo economiza R$5.5M por campanha mantendo o volume de contratos."')

In [ ]:
# PONTUACAO FINAL E DIAGNOSTICO
print('=' * 70)
print('PONTUACAO E DIAGNOSTICO FINAL')
print('=' * 70)
print()
print('DISTRIBUICAO:')
print('  Secao 1 (15 x 0.2pt): 3.0pts')
print('  Secao 2 (10 x 0.5pt): 5.0pts')
print('  Secao 3 (5 x 0.5pt):  2.5pts')
print('  Total:                10.5pts')
print()
print('CRITERIO DE APROVACAO:')
print('  >= 8.0: APROVADO com margem')
print('  >= 6.5: APROVADO (depende do corte)')
print('  < 6.5:  Revisar modulos fracos')
print()
print('ONDE MAIS SE ERRA (baseado na prova de 2019):')
erros = [
    ('Q02', 'R^2 NUNCA diminui -- candidatos escolheram (a)'),
    ('Q04', 'lambda->inf: coefs->zero, NAO infinito -- Q14 prova 2019'),
    ('Q07', 'neg_log_loss retorna negativo -- nao multiplicaram por -1'),
    ('Q08', 'Arvore sem poda tem LL treino ~0 -- Q20 prova 2019'),
    ('Q10', 'Average = media par-a-par, NAO centroide -- Q31 prova 2019'),
    ('Q12', 'Peeking infla falso positivo -- nao conheciam o problema'),
    ('Q15', 'MAPE indefinido com zero -- responderam MAPE=0 -- Q13 prova 2019'),
]
for q, desc in erros:
    print(f'  {q}: {desc}')
print()
print('TEMAS NOVOS EM 2026 vs 2019:')
print('  - Sistemas de recomendacao (Q11, Q13, Q24, Q28)')
print('  - Experimentacao e peeking (Q12, Q25, Q27)')
print('  - Drift e PSI (Q14, Q29, Modulo C)')
print('  - AWS/SageMaker (Modulo C)')
print('  - Comunicacao executiva (Q30)')
print()
print('PROXIMOS PASSOS:')
print('  1. Calcule sua pontuacao neste simulado')
print('  2. Para cada questao errada: volte ao modulo correspondente')
print('  3. Refaca sem olhar o gabarito')
print('  4. Repita o simulado em 1 semana cronometrando 90 minutos')
print('  5. Parta para a Parte 5 (projetos) so com >= 7.0 aqui')